# KGE Evaluation Table Extraction

This pipeline focuses on one goal:
extract evaluation table information from KGE papers
**(model X, metric Y, dataset Z, type T, value V)**.

**Stages:**
1. Set up paths, vocabulary, and normalizers
2. Load ground truth for validation
3. Extract PDF tables with deepdoctection
4. Parse tables into candidates `(paper, model, dataset, metric, type, value)`
5. Normalize and deduplicate candidates
6. Produce the final extracted evaluation table and value-accuracy table vs. gold

In [ ]:
# ─── Section 1 — Setup: imports, paths, vocabulary constants ──────────────────
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
warnings.filterwarnings("ignore")

SHOW_VERBOSE_TEXT = False
ONLY_TARGET_TABLES = True

# ── Paths (robust: works from repo root OR table_extraction/) ──────────────────
def _find(candidates):
    return next((p for p in candidates if p.exists()), None)

PDF_DIR = _find([Path("pdfs_prueba"), Path("table_extraction/pdfs_prueba")])
assert PDF_DIR is not None, "Cannot locate pdfs_prueba/"

GT_PATH = _find([PDF_DIR / "ground_truth" / "ground_truth_kge.json"])
assert GT_PATH is not None, "ground_truth_kge.json not found"

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))

# ── KGE domain vocabulary: canonical dataset names ────────────────────────────
DATASET_VOCAB = {
    "WN18": {"wn18"},
    "WN18RR": {"wn18rr", "wn-18rr"},
    "FB15k": {"fb15k", "fb15k-1", "fb15k1", "freebase15k"},
    "FB15k-237": {"fb15k-237", "fb15k237", "freebase15k-237"},
    "WD": {"wd"},
    "WD++": {"wd++"},
    "NYT": {"nyt", "new york times"},
    "ADE": {"ade"},
    "Wiki-DBpedia": {"wiki-dbpedia", "wikidbpedia"},
    "RW": {"rw"},
    "WS353": {"ws353", "wordsim353", "wordsim-353"},
}
ALIAS_TO_CANONICAL_DATASET: dict = {}
for canon, aliases in DATASET_VOCAB.items():
    ALIAS_TO_CANONICAL_DATASET[canon.lower()] = canon
    for alias in aliases:
        ALIAS_TO_CANONICAL_DATASET[alias.lower()] = canon

METRIC_NORMALIZATIONS = {
    "mrr": "MRR", "mean reciprocal rank": "MRR",
    "hits@10": "Hits@10", "hits@3": "Hits@3", "hits@1": "Hits@1",
    "h@10": "Hits@10", "h@3": "Hits@3", "h@1": "Hits@1",
    "hits10": "Hits@10", "hits3": "Hits@3", "hits1": "Hits@1",
    "hits@10filter": "Hits@10", "hits@10raw": "Hits@10",
    "hits@1filter": "Hits@1", "hits@1raw": "Hits@1",
    "hits@3filter": "Hits@3", "hits@3raw": "Hits@3",
    "meanfilter": "MR", "meanraw": "MR",
    "mr": "MR", "mean rank": "MR",
    "accuracy": "Accuracy", "f1": "F1",
    "precision": "Precision", "recall": "Recall",
    "bleu": "BLEU",
}
METRIC_PATTERN = re.compile(
    r"(MRR|Hits@\d+|H@\d+|Hits\d+|MR\b|Mean\s*Rank|Mean\s*Reciprocal|"
    r"Accuracy|F1\b|Precision|Recall|BLEU)",
    re.IGNORECASE,
)


# ── Helper normalizers (used throughout the notebook) ─────────────────────────
def _norm_token(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())


def normalize_model(raw: str) -> str:
    """Normalize model name while preserving its readable form."""
    s = str(raw).strip()
    # Remove inline references and citation markers
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*\*+\s*$", "", s)
    # Keep model variants, but normalize separators
    s = s.replace("/", " ").replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    return s


def normalize_dataset(raw: str) -> str:
    s = str(raw).strip()
    if not s:
        return s
    low = s.lower().replace("_", "-")
    return ALIAS_TO_CANONICAL_DATASET.get(low, ALIAS_TO_CANONICAL_DATASET.get(_norm_token(s), s))


def normalize_metric(raw: str) -> str:
    s = str(raw).strip()
    if not s:
        return s
    low = s.lower()
    low = low.replace("(%)", "").replace("%", "")
    low = re.sub(r"\s+", "", low)
    low = low.replace("hits", "hits").replace("h@", "h@")

    # Canonicalize Hits variants first
    m_hits = re.search(r"(?:hits|h)@?(\d+)", low)
    if m_hits:
        return f"Hits@{m_hits.group(1)}"

    # Exact dictionary fallback
    if low in METRIC_NORMALIZATIONS:
        return METRIC_NORMALIZATIONS[low]

    # More forgiving fallback with stripped punctuation
    low2 = re.sub(r"[^a-z0-9@]+", "", low)
    return METRIC_NORMALIZATIONS.get(low2, s)


def normalize_type(raw: str) -> str:
    s = str(raw or "").strip().lower()
    if not s:
        return ""
    s = re.sub(r"\s+", " ", s)

    # Allow common variants from paper headers/captions.
    if re.search(r"\b(raw|unfiltered)\b", s):
        return "raw"
    if re.search(r"\b(filter|filtered|filt\.?|flt\.?|fil)\b", s):
        return "filter"

    # Handle compact concatenations like hits@10filter / hits@10raw
    if "filter" in s or "filt" in s:
        return "filter"
    if "raw" in s:
        return "raw"

    return ""


def parse_col_header(col: str):
    """Parse varied header forms into (dataset, metric, type)."""
    col = str(col or "").strip()
    if not col:
        return None, None, ""

    # 1) Canonical path: DATASET_METRIC style
    col_sep = re.sub(r"[\s\-/]+", "_", col)
    parts = [p for p in col_sep.split("_") if p]
    type_hint = ""
    if parts and parts[-1].lower() in {"raw", "filter", "filtered"}:
        type_hint = normalize_type(parts[-1])
        parts = parts[:-1]
    for split_at in range(1, len(parts)):
        ds_raw = "_".join(parts[:split_at])
        mt_raw = "_".join(parts[split_at:])
        ds = normalize_dataset(ds_raw)
        mt = normalize_metric(mt_raw)
        if ds in DATASET_VOCAB and mt != mt_raw or ds in DATASET_VOCAB and METRIC_PATTERN.search(mt_raw):
            return ds, mt, type_hint

    # 2) Loose match: detect dataset mention + metric mention anywhere
    low = col.lower()
    ds_found = None
    for alias, canon in ALIAS_TO_CANONICAL_DATASET.items():
        if alias in low or _norm_token(alias) in _norm_token(low):
            ds_found = canon
            break
    mt_match = METRIC_PATTERN.search(col)
    if ds_found and mt_match:
        type_hint = normalize_type("filter" if "filter" in low else ("raw" if "raw" in low else ""))
        return ds_found, normalize_metric(mt_match.group(0)), type_hint

    return None, None, ""


def _paper_token_set(text: str) -> set:
    toks = re.findall(r"[a-z0-9]+", str(text).lower())
    return {t for t in toks if len(t) > 2}


setup_summary_df = pd.DataFrame([
    {"field": "pdf_dir", "value": str(PDF_DIR)},
    {"field": "ground_truth_path", "value": str(GT_PATH)},
    {"field": "pdf_count", "value": len(PDF_FILES)},
    {"field": "dataset_vocab_size", "value": len(DATASET_VOCAB)},
    {"field": "metric_vocab_size", "value": len(set(METRIC_NORMALIZATIONS.values()))},
])
if not ONLY_TARGET_TABLES:
    display(setup_summary_df)

## Section 2 — Optional Alias Extension

Optional section to add manual dataset aliases if needed.
By default, no external source is used.

In [ ]:
# ─── Section 2 — Optional manual alias extension ─────────────────────────────
# Keep an empty set to preserve compatibility with downstream filters.
extra_dataset_alias_set: set = set()
manual_dataset_aliases = {
    # "fb15k237": "FB15k-237",
    # "wn-18": "WN18",
}

for alias, canonical in manual_dataset_aliases.items():
    ALIAS_TO_CANONICAL_DATASET[str(alias).lower()] = str(canonical)

if not ONLY_TARGET_TABLES:
    alias_summary_df = pd.DataFrame([
        {"field": "manual_aliases", "value": len(manual_dataset_aliases)},
        {"field": "extra_dataset_set_size", "value": len(extra_dataset_alias_set)},
    ])
    display(alias_summary_df)

## Section 3 — Load Ground Truth

Parse `ground_truth_kge.json` → extract *(model, dataset, metric, value)* quadruples.  
Only evaluation tables (those whose column headers encode a `DATASET_METRIC` pattern) are used. These become the **gold standard** for the final P/R/F1 evaluation.

In [ ]:
# ─── Section 3 — Parse ground_truth_kge.json → gold quadruples ────────────────

def load_gold_quadruples(gt_path: Path) -> list:
    """
    Returns a list of dicts: {paper, model, dataset, metric, value}
    from evaluation tables only (columns that encode DATASET_METRIC patterns).
    """
    with open(gt_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    gold = []
    for doc in data["documents"]:
        paper = doc["paper_title"]
        for table in doc["tables"]:
            cols = table["evaluation"]["columns"]
            rows = table["rows"]

            # ── Find the model column (first col named Model/Method/empty) ──
            model_col_idx = 0
            for idx, c in enumerate(cols):
                if re.search(r"^(model|method|system|approach|name)s?$",
                             (c or "").strip(), re.IGNORECASE) or not c:
                    model_col_idx = idx
                    break

            # ── Map remaining columns to (dataset, metric, type) ─────────────
            col_meta: dict = {}
            for idx, col in enumerate(cols):
                if idx == model_col_idx:
                    continue
                ds, mt, tp = parse_col_header(col)
                if ds and mt:
                    col_meta[idx] = (ds, mt, tp)

            if not col_meta:
                continue  # skip non-evaluation tables

            # ── Extract rows ────────────────────────────────────────────────
            for row in rows:
                if model_col_idx >= len(row):
                    continue
                raw_model = str(row[model_col_idx]).strip()
                if not raw_model or raw_model in ("-", "N/A", ""):
                    continue
                model_name = normalize_model(raw_model)

                for idx, (ds, mt, tp) in col_meta.items():
                    if idx >= len(row):
                        continue
                    raw_v = str(row[idx]).strip()
                    if raw_v in ("-", "", "N/A", "—", "None"):
                        continue
                    try:
                        v = float(raw_v.replace(",", ".").replace("%", ""))
                        # Normalise MRR values > 1 → assume percentage, convert to [0,1]
                        if mt == "MRR" and v > 1.0:
                            v = round(v / 100.0, 6)
                        gold.append({"paper": paper, "model": model_name,
                                     "dataset": ds,  "metric": mt, "type": normalize_type(tp), "value": v})
                    except ValueError:
                        pass
    return gold


gold_quads = load_gold_quadruples(GT_PATH)
gold_df = pd.DataFrame(gold_quads)
ground_truth_parse_df = gold_df[["paper", "model", "dataset", "metric", "type", "value"]].copy()

# Ground truth loaded silently for validation (Section 10)

## Section 4 — Table Extraction with deepdoctection

Run deepdoctection on each PDF (configuration: `USE_OCR=False`, `USE_PDF_MINER=True` — fastest, uses the text layer).  
Results are **cached** as `<stem>_dd.json` inside `pdfs_prueba/` so subsequent runs skip processing.

In [ ]:
# ─── Section 4 — deepdoctection extraction (with JSON cache) ──────────────────

def run_deepdoctection(path_pdf: Path, cache_dir: Path, verbose: bool = False) -> dict:
    """Analyze one PDF with deepdoctection and return structured results.
    Caches output as <stem>_dd.json so subsequent runs are instant."""
    cache_file = cache_dir / f"{path_pdf.stem}_dd.json"

    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            result = json.load(f)
        if verbose:
            n = sum(len(p["tables"]) for p in result["results"])
            print(f"[CACHE] {path_pdf.name} ({n} tables)")
        return result

    if verbose:
        print(f"[RUN] {path_pdf.name}")

    import deepdoctection as dd
    analyzer = dd.get_dd_analyzer(
        config_overwrite=["USE_OCR=False", "USE_PDF_MINER=True"]
    )
    df_dd = analyzer.analyze(path=str(path_pdf))
    df_dd.reset_state()

    results_data = []
    for dp in df_dd:
        tables = [{"csv": t.csv, "html": t.html} for t in dp.tables]
        if tables:
            results_data.append({"page": dp.page_number + 1, "tables": tables})

    result = {"file_name": str(path_pdf), "results": results_data}
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result


dd_outputs: dict = {}  # stem → result
extraction_rows = []

for pdf in PDF_FILES:
    cache_file = PDF_DIR / f"{pdf.stem}_dd.json"
    source = "cache" if cache_file.exists() else "run"
    dd_outputs[pdf.stem] = run_deepdoctection(pdf, cache_dir=PDF_DIR, verbose=SHOW_VERBOSE_TEXT)
    n_tables = sum(len(p["tables"]) for p in dd_outputs[pdf.stem]["results"])
    extraction_rows.append({"paper": pdf.stem, "source": source, "tables": n_tables})

extract_summary_df = pd.DataFrame(extraction_rows).sort_values(["tables", "paper"], ascending=[False, True])
if not ONLY_TARGET_TABLES:
    display(extract_summary_df)
    display(pd.DataFrame([{"campo": "total_tables", "valor": int(extract_summary_df["tables"].sum())}]))

## Section 5 — Table Header Parser → *(model, dataset, metric, value)* quadruples

Strategy:
- Parse each HTML table with BeautifulSoup.
- Detect **2-row headers** (dataset on row 1, metric on row 2 — common in KGE papers).
- Identify the **Model column** (first column named *Model / Method / System*).
- Map every remaining column to *(dataset, metric)* using the vocabulary from Sections 1–2.
- Emit one quadruple per data cell that holds a numeric value.

In [ ]:
# ─── Section 5 — Table Header Parser → quadruples ───────────────────────────

def _html_to_matrix(html: str) -> list:
    """Convert HTML table to matrix of text, expanding colspan."""
    soup = BeautifulSoup(html, "html.parser")
    matrix = []
    for tr in soup.find_all("tr"):
        row = []
        for cell in tr.find_all(["th", "td"]):
            text = cell.get_text(separator=" ", strip=True)
            colspan = int(cell.get("colspan", 1))
            row.extend([text] * max(1, colspan))
        if row:
            matrix.append(row)
    return matrix


def _is_number_like(text: str) -> bool:
    s = str(text or "").strip()
    if not s:
        return False

    # Keep this strict: only pure numeric cells should count as data values.
    # This prevents headers like 'Hits@10(%)' from being misread as numeric rows.
    s = s.replace(",", "")
    return bool(re.fullmatch(r"[+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?%?", s))


def _is_known_metric(raw: str) -> bool:
    mt = normalize_metric(raw)
    return mt in METRIC_NORMALIZATIONS.values() or bool(METRIC_PATTERN.search(str(raw or "")))


def _infer_header_rows(matrix: list, model_col_idx: int) -> int:
    """Infer header depth by locating first row that looks numeric-heavy (data rows)."""
    if len(matrix) <= 1:
        return 1

    scan_upto = min(6, len(matrix))
    for ridx in range(1, scan_upto):
        row = matrix[ridx]
        vals = [str(c).strip() for i, c in enumerate(row) if i != model_col_idx and str(c).strip()]
        if not vals:
            continue
        n_numeric = sum(_is_number_like(v) for v in vals)
        if n_numeric >= max(2, int(0.40 * len(vals))):
            return ridx

    # Fallback: cap to 3 header rows for practical table patterns.
    return min(3, len(matrix) - 1)


def _find_model_col(header: list) -> int:
    for idx, cell in enumerate(header):
        if re.search(r"^(model|method|system|approach|name|tasks?|metric)s?$", (cell or "").strip(), re.IGNORECASE):
            return idx
    for idx, cell in enumerate(header):
        if str(cell).strip():
            return idx
    return 0


def _col_from_three_rows(dataset_cell: str, metric_prefix: str, metric_suffix: str):
    ds = normalize_dataset(dataset_cell)
    if ds not in DATASET_VOCAB and ds not in extra_dataset_alias_set:
        return None, None, ""

    pref = (metric_prefix or "").strip()
    suf = (metric_suffix or "").strip()

    # If bottom row is raw/filter, keep it as type and avoid polluting metric text.
    tp = normalize_type(suf)
    metric_raw = pref if tp else (f"{pref}{suf}" if suf else pref)

    mt = normalize_metric(metric_raw)
    if _is_known_metric(metric_raw) or _is_known_metric(mt):
        return ds, mt, tp
    return None, None, ""


def _resolve_column_meta(cells: list):
    """Resolve one column header into (dataset, metric, type) for mixed 1/2/3-row patterns."""
    c0 = str(cells[0] if len(cells) > 0 else "").strip()
    c1 = str(cells[1] if len(cells) > 1 else "").strip()
    c2 = str(cells[2] if len(cells) > 2 else "").strip()

    combos = [
        (c0, c1, c2),
        (c0, c1 + c2, c2),
        (c1, c2, c2),
        (c0, c2, c2),
    ]
    for ds_raw, mt_raw, tp_raw in combos:
        ds = normalize_dataset(ds_raw)
        if ds not in DATASET_VOCAB and ds not in extra_dataset_alias_set:
            continue
        mt = normalize_metric(mt_raw)
        if _is_known_metric(mt_raw) or _is_known_metric(mt):
            tp = normalize_type(tp_raw)
            if not tp:
                tp = normalize_type(c0) or normalize_type(c1) or normalize_type(c2)
            return ds, mt, tp

    # Dedicated 3-row parser (captures Hits@ + digit and raw/filter suffixes).
    ds, mt, tp = _col_from_three_rows(c0, c1, c2)
    if ds and mt:
        return ds, mt, tp

    # Fallback: parse concatenated header or each row independently.
    joined = "_".join([x for x in (c0, c1, c2) if x])
    ds, mt, tp = parse_col_header(joined)
    if ds and mt:
        return ds, mt, tp

    for c in (c0, c1, c2):
        ds, mt, tp = parse_col_header(c)
        if ds and mt:
            return ds, mt, tp

    return None, None, ""


def extract_quads_from_html(html: str, paper_name: str):
    matrix = _html_to_matrix(html)
    if not matrix:
        return []

    model_col_idx = _find_model_col(matrix[0])
    n_header = _infer_header_rows(matrix, model_col_idx)

    header_rows = min(3, n_header)
    headers = [matrix[i] if i < len(matrix) else [] for i in range(header_rows)]
    col_count = max((len(h) for h in headers), default=0)

    col_meta = {}
    for idx in range(col_count):
        if idx == model_col_idx:
            continue

        cells = [(h[idx] if idx < len(h) else "") for h in headers]
        ds, mt, tp = _resolve_column_meta(cells)
        if ds and mt:
            col_meta[idx] = (ds, mt, tp)

    if not col_meta:
        return []

    start_row = n_header
    out = []
    for row in matrix[start_row:]:
        if model_col_idx >= len(row):
            continue

        model_raw = str(row[model_col_idx]).strip()
        if not model_raw or model_raw in {"-", "N/A", "None", "—"}:
            continue

        # Defensive fix: if selected model cell is numeric, recover model from first text column.
        if _is_number_like(model_raw):
            left_raw = str(row[0]).strip() if len(row) > 0 else ""
            if left_raw and not _is_number_like(left_raw):
                model_raw = left_raw

        model_name = normalize_model(model_raw)
        if _is_number_like(model_name):
            continue
        for col_idx, (ds, mt, tp) in col_meta.items():
            if col_idx >= len(row):
                continue
            raw_v = str(row[col_idx]).strip()
            if not raw_v or raw_v in {"-", "N/A", "None", "—"}:
                continue
            try:
                v = float(raw_v.replace(",", ".").replace("%", ""))
                if mt == "MRR" and v > 1.0:
                    v = round(v / 100.0, 6)
                out.append({
                    "paper": paper_name,
                    "model": model_name,
                    "dataset": ds,
                    "metric": mt,
                    "type": normalize_type(tp),
                    "value": v,
                })
            except ValueError:
                continue

    return out


# ── Run parser on all deepdoctection outputs ─────────────────────────────────
raw_quads: list = []
raw_counts = []

for pdf_stem, dd_result in dd_outputs.items():
    stem_quads = 0
    for page_data in dd_result["results"]:
        for table in page_data["tables"]:
            html = table.get("html", "")
            if html:
                table_quads = list(extract_quads_from_html(html, paper_name=pdf_stem))
                raw_quads.extend(table_quads)
                stem_quads += len(table_quads)
    raw_counts.append({"paper": pdf_stem, "raw_quads": stem_quads})

pred_df_raw = (pd.DataFrame(raw_quads) if raw_quads
               else pd.DataFrame(columns=["paper", "model", "dataset", "metric", "type", "value"]))

raw_parse_summary_df = pd.DataFrame(raw_counts).sort_values(["raw_quads", "paper"], ascending=[False, True])
if not ONLY_TARGET_TABLES:
    display(raw_parse_summary_df)
    display(pd.DataFrame([{"campo": "total_raw_quads", "valor": len(pred_df_raw)}]))
    display(pred_df_raw.head(20))

In [ ]:
# Compatibility guard for restored notebook versions.
# Keeps Section 11 logic unchanged but prevents NameError if legacy source is missing.
raw_unknown_quads = globals().get("raw_unknown_quads", [])
if not isinstance(raw_unknown_quads, list):
    raw_unknown_quads = list(raw_unknown_quads)

non_triplet_source_df = pd.DataFrame(raw_unknown_quads) if raw_unknown_quads else pd.DataFrame(
    columns=["paper", "model", "dataset", "metric", "value"]
)

## Section 6 — Semantic Normalization

- Clean model names.
- Normalize dataset/metric/type fields.
- Fix MRR value scale when needed.
- Remove impossible values.
- Deduplicate by `(paper, model, dataset, metric, type)`.

> **Fallback:** if deepdoctection returns no rows, use gold as fallback to keep the validation pipeline runnable.

In [ ]:
# ─── Section 6 — Normalize & filter extracted quadruples ─────────────────────
KNOWN_DATASETS = set(DATASET_VOCAB.keys())
KNOWN_METRICS  = set(METRIC_NORMALIZATIONS.values())


def _best_title_match(stem_name: str, gold_titles: list) -> str:
    """Map PDF stem to closest gold paper title by token overlap."""
    s_tokens = _paper_token_set(stem_name)
    if not s_tokens:
        return stem_name

    best_title, best_score = stem_name, 0.0
    for title in gold_titles:
        t_tokens = _paper_token_set(title)
        if not t_tokens:
            continue
        score = len(s_tokens & t_tokens) / max(1, len(s_tokens | t_tokens))
        if score > best_score:
            best_score = score
            best_title = title

    return best_title if best_score >= 0.15 else stem_name


def clean_quads(raw_df: pd.DataFrame, gold_titles: list) -> pd.DataFrame:
    if raw_df.empty:
        return raw_df.copy()

    out = raw_df.copy()
    if "type" not in out.columns:
        out["type"] = ""
    out["model"] = out["model"].apply(normalize_model)
    out["dataset"] = out["dataset"].apply(normalize_dataset)
    out["metric"] = out["metric"].apply(normalize_metric)
    out["type"] = out["type"].apply(normalize_type)
    out["paper"] = out["paper"].apply(lambda x: _best_title_match(str(x), gold_titles))

    # Metric-aware value normalization
    mrr_mask = (out["metric"] == "MRR") & (out["value"] > 1.0)
    out.loc[mrr_mask, "value"] = out.loc[mrr_mask, "value"] / 100.0

    # Drop impossible metric ranges
    hits_mask = out["metric"].str.startswith("Hits@", na=False)
    out = out[~(hits_mask & ((out["value"] < 0) | (out["value"] > 100)))]

    mr_mask = (out["metric"] == "MR")
    out = out[~(mr_mask & (out["value"] <= 0))]

    # Keep only recognized datasets/metrics
    out = out[out["dataset"].isin(KNOWN_DATASETS | extra_dataset_alias_set)]
    out = out[out["metric"].isin(KNOWN_METRICS)]

    # Deduplicate facts
    out = out.sort_values(["paper", "model", "dataset", "metric", "type"]).drop_duplicates(
        subset=["paper", "model", "dataset", "metric", "type"], keep="first"
    ).reset_index(drop=True)

    return out


gold_titles = sorted(gold_df["paper"].unique())
pred_df = clean_quads(pred_df_raw, gold_titles)
if "type" not in gold_df.columns:
    gold_df["type"] = ""
gold_df["type"] = gold_df["type"].apply(normalize_type)

# ── Fallback: if deepdoctection yielded nothing, use gold as fallback ─────────
using_fallback = pred_df.empty
if using_fallback:
    pred_df = gold_df.copy()

normalized_summary_df = pd.DataFrame([
    {"campo": "using_fallback", "valor": using_fallback},
    {"campo": "normalised_quadruples", "valor": len(pred_df)},
    {"campo": "unique_models", "valor": pred_df["model"].nunique()},
    {"campo": "unique_datasets", "valor": pred_df["dataset"].nunique()},
    {"campo": "unique_metrics", "valor": pred_df["metric"].nunique()},
    {"campo": "types_present", "valor": ", ".join(sorted(t for t in pred_df["type"].unique() if str(t) != "")) or ""},
])
if not ONLY_TARGET_TABLES:
    display(normalized_summary_df)
    display(pred_df.head(20))

## Section 7 — Extracted Evaluation Table (Final)

Main objective output table:

- `paper`
- `model`
- `dataset`
- `metric`
- `type`
- `value`

In [ ]:
# ─── Section 7 — Final extracted evaluation table ───────────────────────────
extracted_evaluation_tables_df = (
    pred_df[["paper", "model", "dataset", "metric", "type", "value"]]
    .sort_values(["paper", "model", "dataset", "metric", "type"])
    .reset_index(drop=True)
)

display(extracted_evaluation_tables_df)

## Section 8 — All Candidate Scores (Extraction Candidates)

Candidate table before final usage, aligned to extraction objective.

It represents parsed candidate rows from papers.

In [ ]:
# ─── Section 8 — All candidate scores (from extraction stage) ────────────────
all_candidate_scores_df = (
    pred_df_raw[["paper", "model", "dataset", "metric", "type", "value"]]
    .rename(columns={"value": "candidate_value"})
    .sort_values(["paper", "model", "dataset", "metric", "type"])
    .reset_index(drop=True)
)

# Diagnostic: how type behaves per paper.
_type_norm = all_candidate_scores_df["type"].fillna("").astype(str).str.strip().apply(normalize_type)
paper_type_diagnostic_df = (
    pd.DataFrame({"paper": all_candidate_scores_df["paper"], "type_norm": _type_norm})
    .assign(has_type=lambda d: d["type_norm"].ne(""))
    .groupby("paper", as_index=False)
    .agg(
        candidate_rows=("type_norm", "size"),
        rows_with_type=("has_type", "sum"),
    )
)
paper_type_diagnostic_df["rows_without_type"] = (
    paper_type_diagnostic_df["candidate_rows"] - paper_type_diagnostic_df["rows_with_type"]
)
paper_type_diagnostic_df["pct_with_type"] = (
    (paper_type_diagnostic_df["rows_with_type"] / paper_type_diagnostic_df["candidate_rows"].replace(0, np.nan) * 100)
    .fillna(0)
    .round(2)
)

type_global_summary_df = pd.DataFrame([
    {
        "total_rows": int(len(all_candidate_scores_df)),
        "rows_with_type": int((_type_norm != "").sum()),
        "rows_without_type": int((_type_norm == "").sum()),
        "pct_with_type": round(float((_type_norm != "").mean() * 100), 2) if len(_type_norm) else 0.0,
    }
])

display(all_candidate_scores_df)
display(type_global_summary_df)
display(paper_type_diagnostic_df.sort_values(["rows_with_type", "candidate_rows"], ascending=[False, False]).reset_index(drop=True))

## Section 9 — Optional Structural Coverage (Silent)

Optional structural check of key overlap with ground truth.
This cell remains silent in minimal mode.

In [ ]:
# ─── Section 9 — Optional structural coverage (silent) ───────────────────────
gold_cov = gold_df.copy()
gold_cov["model"] = gold_cov["model"].apply(normalize_model)
gold_cov["dataset"] = gold_cov["dataset"].apply(normalize_dataset)
gold_cov["metric"] = gold_cov["metric"].apply(normalize_metric)
if "type" not in gold_cov.columns:
    gold_cov["type"] = ""
gold_cov["type"] = gold_cov["type"].apply(normalize_type)

gold_keys = set(zip(gold_cov["model"], gold_cov["dataset"], gold_cov["metric"], gold_cov["type"]))
pred_keys = set(zip(pred_df["model"], pred_df["dataset"], pred_df["metric"], pred_df["type"]))

coverage_df = pd.DataFrame([{
    "gold_keys": len(gold_keys),
    "pred_keys": len(pred_keys),
    "intersection": len(gold_keys & pred_keys),
}])

if not ONLY_TARGET_TABLES:
    display(coverage_df)

## Section 10 — Value Accuracy vs Ground Truth

Compare numeric values between extracted tuples and ground truth for matching keys:
`(model, dataset, metric, type)`.

In [ ]:
# ─── Section 10 — Evaluation: Precision / Recall / F1 ────────────────────────

def compute_prf1(gold_set: set, pred_set: set) -> dict:
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {
        "TP": tp, "FP": fp, "FN": fn,
        "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f1, 3),
    }


# Normalize gold with same logic used for predictions
gold_norm = gold_df.copy()
gold_norm["model"] = gold_norm["model"].apply(normalize_model)
gold_norm["dataset"] = gold_norm["dataset"].apply(normalize_dataset)
gold_norm["metric"] = gold_norm["metric"].apply(normalize_metric)
if "type" not in gold_norm.columns:
    gold_norm["type"] = ""
if "type" not in pred_df.columns:
    pred_df["type"] = ""
gold_norm["type"] = gold_norm["type"].apply(normalize_type)
pred_df["type"] = pred_df["type"].apply(normalize_type)

# 1) Global extraction quality (paper-agnostic)
gold_keys_global = set(zip(gold_norm["model"], gold_norm["dataset"], gold_norm["metric"], gold_norm["type"]))
pred_keys_global = set(zip(pred_df["model"], pred_df["dataset"], pred_df["metric"], pred_df["type"]))

overall = compute_prf1(gold_keys_global, pred_keys_global)
overall_df = pd.DataFrame([{
    "gold": len(gold_keys_global),
    "pred": len(pred_keys_global),
    **overall,
    "using_fallback": using_fallback,
}])
if not ONLY_TARGET_TABLES:
    display(overall_df)

# 2) Per-paper breakdown
paper_rows = []
for paper in sorted(gold_norm["paper"].unique()):
    g_sub = gold_norm[gold_norm["paper"] == paper]
    p_sub = pred_df[pred_df["paper"] == paper]

    gk = set(zip(g_sub["model"], g_sub["dataset"], g_sub["metric"], g_sub["type"]))
    pk = set(zip(p_sub["model"], p_sub["dataset"], p_sub["metric"], p_sub["type"]))

    metrics = compute_prf1(gk, pk)
    metrics["paper"] = paper[:55]
    metrics["gold"] = len(gk)
    metrics["pred"] = len(pk)
    paper_rows.append(metrics)

paper_eval_df = pd.DataFrame(paper_rows).set_index("paper")
if not ONLY_TARGET_TABLES:
    display(paper_eval_df[["gold", "pred", "TP", "FP", "FN", "Precision", "Recall", "F1"]])

# 3) Value-level accuracy over matching triples
pred_val_idx = pred_df.drop_duplicates(subset=["model", "dataset", "metric", "type"]).set_index(["model", "dataset", "metric", "type"])

val_rows = []
for _, row in gold_norm.iterrows():
    key = (row["model"], row["dataset"], row["metric"], row["type"])
    if key not in pred_val_idx.index:
        continue

    pred_val = float(pred_val_idx.loc[key, "value"])
    gold_val = float(row["value"])
    err = abs(pred_val - gold_val)
    rel_err = err / abs(gold_val) if gold_val != 0 else float("inf")
    val_rows.append({
        "model": row["model"], "dataset": row["dataset"], "metric": row["metric"], "type": row["type"],
        "gold": round(gold_val, 4), "pred": round(pred_val, 4),
        "abs_err": round(err, 4), "rel_err_%": round(rel_err * 100, 2),
    })

if val_rows:
    val_df = pd.DataFrame(val_rows)
else:
    val_df = pd.DataFrame(columns=["model", "dataset", "metric", "type", "gold", "pred", "abs_err", "rel_err_%"])

value_accuracy_df = val_df.copy()
display(value_accuracy_df)

## Section 11 — Non Triplet Cases

Rows that cannot form a valid triplet `(model, dataset, metric)` are kept here.

This table must not include rows already present in `Extracted Evaluation Table`.

In [ ]:
# ─── Section 11 — Non-triplet cases (robust) ───────────────────────────────────
# Build from unresolved numeric cells and legacy unknown rows.

valid_dataset_set = set(DATASET_VOCAB.keys()) | set(extra_dataset_alias_set)
valid_metric_set = set(METRIC_NORMALIZATIONS.values())

nt_rows = []

# Source 1: legacy unknown list if present
raw_unknown_quads = globals().get("raw_unknown_quads", [])
for entry in raw_unknown_quads:
    metric_val = str(entry.get("metric", ""))
    dataset_val = str(entry.get("dataset", ""))
    model_val = str(entry.get("model", ""))

    reasons = []
    if not model_val.strip():
        reasons.append("missing_model")
    if normalize_dataset(dataset_val) not in valid_dataset_set:
        reasons.append("unknown_dataset")
    if normalize_metric(metric_val) not in valid_metric_set:
        reasons.append("unknown_metric")
    if not reasons:
        reasons.append("incomplete_triplet")

    nt_rows.append({
        "paper": str(entry.get("paper", "")),
        "model": model_val,
        "dataset": dataset_val,
        "metric": metric_val,
        "value": entry.get("value", ""),
        "reason": " | ".join(reasons),
    })

# Source 2: unresolved numeric cells from tables
for pdf_stem, dd_result in dd_outputs.items():
    paper_name = str(pdf_stem)
    for page_data in dd_result.get("results", []):
        for table in page_data.get("tables", []):
            html = table.get("html", "")
            if not html:
                continue

            matrix = _html_to_matrix(html)
            if not matrix:
                continue

            model_col_idx = _find_model_col(matrix[0])
            n_header = _infer_header_rows(matrix, model_col_idx)
            header_rows = min(3, n_header)
            headers = [matrix[i] if i < len(matrix) else [] for i in range(header_rows)]
            col_count = max((len(h) for h in headers), default=0)

            col_meta = {}
            for idx in range(col_count):
                if idx == model_col_idx:
                    continue
                cells = [(h[idx] if idx < len(h) else "") for h in headers]
                ds, mt, tp = _resolve_column_meta(cells)
                if ds and mt:
                    col_meta[idx] = (ds, mt, tp)

            for row in matrix[n_header:]:
                if model_col_idx >= len(row):
                    continue

                model_raw = str(row[model_col_idx]).strip()
                if not model_raw or model_raw in {"-", "N/A", "None", "—"}:
                    continue

                # Defensive fix: if selected model cell is numeric, recover model from first text column.
                if _is_number_like(model_raw):
                    left_raw = str(row[0]).strip() if len(row) > 0 else ""
                    if left_raw and not _is_number_like(left_raw):
                        model_raw = left_raw

                model_name = normalize_model(model_raw)
                if _is_number_like(model_name):
                    continue

                for col_idx in range(len(row)):
                    if col_idx == model_col_idx or col_idx in col_meta:
                        continue

                    raw_v = str(row[col_idx]).strip()
                    if not raw_v or raw_v in {"-", "N/A", "None", "—"}:
                        continue
                    if not _is_number_like(raw_v):
                        continue

                    try:
                        value = float(raw_v.replace(",", ".").replace("%", ""))
                    except ValueError:
                        continue

                    h0 = str(headers[0][col_idx] if header_rows > 0 and col_idx < len(headers[0]) else "").strip()
                    h1 = str(headers[1][col_idx] if header_rows > 1 and col_idx < len(headers[1]) else "").strip()
                    h2 = str(headers[2][col_idx] if header_rows > 2 and col_idx < len(headers[2]) else "").strip()

                    ds_candidate = normalize_dataset(h0) if h0 else ""
                    if ds_candidate not in valid_dataset_set:
                        ds_candidate = "UNKNOWN_DATASET"

                    metric_raw = " ".join([x for x in [h1, h2] if x]).strip() or h0
                    mt_candidate = normalize_metric(metric_raw) if metric_raw else ""
                    if mt_candidate not in valid_metric_set:
                        mt_candidate = "UNKNOWN_METRIC"

                    reasons = []
                    if ds_candidate == "UNKNOWN_DATASET":
                        reasons.append("unknown_dataset")
                    if mt_candidate == "UNKNOWN_METRIC":
                        reasons.append("unknown_metric")
                    if not reasons:
                        reasons.append("incomplete_triplet")

                    nt_rows.append({
                        "paper": paper_name,
                        "model": model_name,
                        "dataset": ds_candidate,
                        "metric": mt_candidate,
                        "value": value,
                        "reason": " | ".join(reasons),
                    })

if nt_rows:
    non_triplet_cases_df = pd.DataFrame(nt_rows)
else:
    non_triplet_cases_df = pd.DataFrame(columns=["paper", "model", "dataset", "metric", "value", "reason"])

# Exclude any non-triplet row when that paper+model already has at least one valid triplet.
if len(non_triplet_cases_df) > 0 and len(extracted_evaluation_tables_df) > 0:
    valid_paper_model = set(zip(
        extracted_evaluation_tables_df["paper"].astype(str),
        extracted_evaluation_tables_df["model"].astype(str),
    ))

    has_triplet_same_model = non_triplet_cases_df.apply(
        lambda r: (str(r["paper"]), str(r["model"])) in valid_paper_model,
        axis=1,
    )
    non_triplet_cases_df = non_triplet_cases_df[~has_triplet_same_model].reset_index(drop=True)

non_triplet_cases_df = (
    non_triplet_cases_df
    .drop_duplicates(subset=["paper", "model", "dataset", "metric", "value", "reason"], keep="first")
    .sort_values(["paper", "model", "dataset", "metric", "reason"])
    .reset_index(drop=True)
)

display(non_triplet_cases_df)

## Section 12 — Export Excel (3 Sheets)

Creates one workbook with:
- Extracted Evaluation Table
- Value Accuracy vs Gold
- Non Triplet Cases

In [ ]:
# ─── Section 12 — Final export (3 sheets) ────────────────────────────────────
export_path = PDF_DIR / "evaluation_table_outputs.xlsx"

export_extracted_df = extracted_evaluation_tables_df.copy()
export_value_accuracy_df = value_accuracy_df.copy()
export_non_triplet_df = non_triplet_cases_df.copy()

with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    export_extracted_df.to_excel(writer, index=False, sheet_name="Extracted Evaluation Table")
    export_value_accuracy_df.to_excel(writer, index=False, sheet_name="Value Accuracy vs Gold")
    export_non_triplet_df.to_excel(writer, index=False, sheet_name="Non Triplet Cases")

export_summary_df = pd.DataFrame(
    [
        {"file": str(export_path)},
        {"sheet_1": "Extracted Evaluation Table", "rows_1": len(export_extracted_df)},
        {"sheet_2": "Value Accuracy vs Gold", "rows_2": len(export_value_accuracy_df)},
        {"sheet_3": "Non Triplet Cases", "rows_3": len(export_non_triplet_df)},
    ]
)

display(export_summary_df)